# Chord Comparison

A "chord" = multiple(>=3) notes with the same or very close start times
- Block Chords: multiple(>=3) notes with the same start time
- Broken Chords / Arpeggios: multiple(>=3) notes play one after another, in a given order

Chord comparison can be considered as a set matching problem so that the correctness of a chord can be determined by the overlap between response set and reference set. Additionally, it may require handling partial matches problem (e.g. two out of three chord notes played correctly), consider defining it as imperfect chords. 

In [1]:
import os
import json
import numpy as np

cwd = os.getcwd()

cwd = os.getcwd()
dir = os.path.dirname(cwd)
reference_path = os.path.join(dir, "data", "referenceMIDIchords.json")
response_path = os.path.join(dir, "data", "responseMIDIchords.json")

with open(reference_path) as f1:
    ref_chords_data = json.load(f1)

with open(response_path) as f2:
    res_chords_data = json.load(f2)

chord accuracy metric: https://arxiv.org/pdf/2201.05244

In [ ]:
# Default threshold: notes starting within 50ms are grouped as one chord.
DEFAULT_CHORD_ONSET_WINDOW = 0.05

# Chord template dictionary.
# Each entry maps a chord name to a frozenset of pitch class intervals,
# where the root note is normalised to 0.
# Only the 4 triad types are included, following Muller (2021) Chapter 5.
CHORD_TEMPLATES = {
    "major":      frozenset([0, 4, 7]),
    "minor":      frozenset([0, 3, 7]),
    "diminished": frozenset([0, 3, 6]),
    "augmented":  frozenset([0, 4, 8]),
}
 
# Pitch class names for human-readable feedback messages.
PITCH_CLASS_NAMES = ["C", "C#", "D", "D#", "E", "F",
                     "F#", "G", "G#", "A", "A#", "B"]

# helper function to build an event dict from a group of notes
def make_event(notes_in_group):
    """
    Build a single event dict from a group of notes.
 
    Args:
        notes_in_group: list of one or more note dicts.
 
    Returns:
        event dict with keys "event_type", "notes", "start". example:
        {
            "event_type": "note" or "chord" depending on the number of notes in the group
            "notes": [                    
                {                         
                    "pitch": int
                    "start": float
                    "duration": float
                },
            ]
            "start": float,
        }
    """
    event_type = "note" if len(notes_in_group) == 1 else "chord"
    return {
        "event_type": event_type,
        "notes": notes_in_group,
        "start": notes_in_group[0]["start"],
    }

# group notes into events based on their start times
def group_notes_into_events(notes, chord_onset_window=DEFAULT_CHORD_ONSET_WINDOW):
    """
    Group a flat list of notes into events. Notes whose start times fall
    within chord_onset_window seconds of each other are placed into the
    same event (i.e. treated as a chord). Notes that are not grouped with
    any other note form a single-note event.

    Args:
        notes: list of dicts, each with keys 'pitch', 'start', 'duration'
        chord_onset_window: float, max time difference (seconds) to be grouped

    Returns:
        list of dicts, each with keys:
            'pitches'  : set of int MIDI pitch numbers
            'start'    : float, earliest start time in the group
            'duration' : float, average duration of notes in the group
    """
    if len(notes) == 0:
        return []

    # Sort notes by start time first
    sorted_notes = sorted(notes, key=lambda n: n["start"])

    events = []
    current_group = [sorted_notes[0]]
    group_start = sorted_notes[0]["start"]

    for note in sorted_notes[1:]:
        # Close enough in time: add to current group (chord)
        if note["start"] - group_start <= chord_onset_window:
            current_group.append(note)
        else:
            # Too far apart: save current chord, start a new group
            event = make_event(current_group)
            events.append(event)
            current_group = [note]
            group_start = note["start"]

    # append the last group
    last_event = make_event(current_group)
    events.append(last_event)

    return events


def compute_event_cost(event_a, event_b):
    pass